In [171]:
# Only for preprocessing!
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [173]:
df = pd.read_csv('ultimate_combined_speeches.csv')


stop_words = {'the', 'and', 'to', 'of', 'in', 'that', 'we', 'is', 'a', 'it', 
              'for', 'our', 'on', 'this', 'are', 'i', 'have', 'with', 'will', 
              'you', 'not', 'be', 'as', 'but', 'can', 'my', 'has', 'their', 'they', 'which', 'from', 'people', 'been', 'president', 'mr', 'sates'}


def remove_stop_words(text):
    words = str(text).split()
    clean_words = [w for w in words if w not in stop_words]
    return " ".join(clean_words)

df['Text'] = df['Text'].apply(remove_stop_words)

In [205]:
speeches = df['Text'].astype(str).values
labels = df['Label'].values

#preprocessing
max_vocab = 500
tokenizer = Tokenizer(num_words=max_vocab)
tokenizer.fit_on_texts(speeches) 
X_numeric = tokenizer.texts_to_sequences(speeches)

#Padding
max_speech_length = 500  
X_padded = pad_sequences(X_numeric, maxlen=max_speech_length, padding='post', truncating='post')

#Reshape y into a column vector
y_array = np.array(labels).reshape(-1, 1)

# 80/20 Split
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, 
    y_array, 
    test_size=0.2, 
    random_state=42
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (1248, 500)
Testing data shape: (313, 500)


In [206]:
# https://cs231n.github.io/neural-networks-3/#ada used as a guide
def adam_update(param, grad, m, v, t, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    """Updates a single weight matrix using Adam optimization."""
    m[:] = beta1 * m + (1 - beta1) * grad
    v[:] = beta2 * v + (1 - beta2) * grad ** 2
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    param -= lr * m_hat / (np.sqrt(v_hat) + eps)

#layers
class Embedding:
    def __init__(self, vocab_size, embed_dim):
        self.W = np.random.randn(vocab_size, embed_dim) * 0.1
        self.mW = np.zeros_like(self.W); self.vW = np.zeros_like(self.W)
        self.t = 0

    def forward(self, X):
        self.X = X
        return self.W[X]

    def backward(self, d_out, lr):
        self.t += 1
        dW = np.zeros_like(self.W)
        np.add.at(dW, self.X, d_out)
        adam_update(self.W, dW, self.mW, self.vW, self.t, lr)
        return None

class Conv1D:
    def __init__(self, in_channels, out_channels, kernel_size):
        self.kernel_size = kernel_size
        self.W = np.random.randn(kernel_size, in_channels, out_channels) * np.sqrt(2.0 / (kernel_size * in_channels))
        self.b = np.zeros((out_channels,))
        
        self.mW, self.vW = np.zeros_like(self.W), np.zeros_like(self.W)
        self.mb, self.vb = np.zeros_like(self.b), np.zeros_like(self.b)
        self.t = 0

    def forward(self, X):
        self.X = X
        batch_size, seq_len, _ = X.shape
        self.out_seq_len = seq_len - self.kernel_size + 1
        output = np.zeros((batch_size, self.out_seq_len, self.b.shape[0]))
        
        for i in range(self.out_seq_len):
            patch = X[:, i:i + self.kernel_size, :]
            output[:, i, :] = np.tensordot(patch, self.W, axes=([1, 2], [0, 1])) + self.b
        return output

    def backward(self, d_out, lr):
        self.t += 1
        dW = np.zeros_like(self.W)
        db = np.sum(d_out, axis=(0, 1))
        dX = np.zeros_like(self.X)
        
        for i in range(self.out_seq_len):
            patch = self.X[:, i:i + self.kernel_size, :]
            dW += np.tensordot(patch, d_out[:, i, :], axes=([0], [0]))
            dX[:, i:i + self.kernel_size, :] += np.tensordot(d_out[:, i, :], self.W, axes=([1], [2]))
            
        adam_update(self.W, dW, self.mW, self.vW, self.t, lr)
        adam_update(self.b, db, self.mb, self.vb, self.t, lr)
        return dX

class Dense:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * np.sqrt(2.0 / in_features)
        self.b = np.zeros((1, out_features))
        
        self.mW, self.vW = np.zeros_like(self.W), np.zeros_like(self.W)
        self.mb, self.vb = np.zeros_like(self.b), np.zeros_like(self.b)
        self.t = 0

    def forward(self, X):
        self.X = X
        return np.dot(X, self.W) + self.b

    def backward(self, d_out, lr):
        self.t += 1
        dX = np.dot(d_out, self.W.T)
        dW = np.dot(self.X.T, d_out)
        db = np.sum(d_out, axis=0, keepdims=True)
        
        adam_update(self.W, dW, self.mW, self.vW, self.t, lr)
        adam_update(self.b, db, self.mb, self.vb, self.t, lr)
        return dX

class GlobalMaxPool1D:
    def forward(self, X):
        self.X = X
        self.argmax = np.argmax(X, axis=1)
        return np.max(X, axis=1)

    def backward(self, d_out, lr):
        batch_size, seq_len, channels = self.X.shape
        dX = np.zeros_like(self.X)
        b_idx = np.arange(batch_size)[:, None]
        c_idx = np.arange(channels)[None, :]
        dX[b_idx, self.argmax, c_idx] = d_out
        return dX

class ReLU:
    def forward(self, X):
        self.X = X
        return np.maximum(0, X)

    def backward(self, d_out, lr):
        return d_out * (self.X > 0)

class Dropout:
    def __init__(self, rate=0.5):
        self.rate = rate
        self.training = True

    def forward(self, X):
        if not self.training: return X
        self.mask = (np.random.rand(*X.shape) > self.rate) / (1 - self.rate)
        return X * self.mask

    def backward(self, d_out, lr):
        return d_out * self.mask

class Sigmoid:
    def forward(self, X):
        self.out = 1 / (1 + np.exp(-np.clip(X, -250, 250)))
        return self.out

    def backward(self, d_out, lr):
        return d_out * self.out * (1 - self.out)

class BinaryCrossEntropy:
    def forward(self, y_pred, y_true):
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    def backward(self, y_pred, y_true):
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        return (y_pred - y_true) / (y_pred * (1 - y_pred)) / y_true.shape[0]

In [211]:
#training
def build_layers():
    return [
        Embedding(500, 50),
        Conv1D(50, 64, 3),
        ReLU(),
        Conv1D(64, 128, 3),
        ReLU(),
        GlobalMaxPool1D(),
        Dropout(0.4),
        Dense(128, 1),
        Sigmoid(),]

def set_training_mode(layers, training: bool):
    for layer in layers:
        if isinstance(layer, Dropout):
            layer.training = training

def basic_fit(X_train, y_train, X_test, y_test, layers, epochs=15, lr=0.001, batch_size=32):
    loss_fn = BinaryCrossEntropy()
    n = X_train.shape[0]   
    # storing
    train_acc_history = []
    test_acc_history = []   
    for epoch in range(epochs):        
        #shuffle the data
        idx = np.random.permutation(n)
        X_s, y_s = X_train[idx], y_train[idx]
        set_training_mode(layers, training=True)         
        for i in range(0, n, batch_size):
            Xb = X_s[i:i + batch_size]
            yb = y_s[i:i + batch_size]
            out = Xb
            for layer in layers:
                out = layer.forward(out)        
            loss = loss_fn.forward(out, yb)      
            d = loss_fn.backward(out, yb)
            for layer in reversed(layers):
                d = layer.backward(d, lr)               
        #training acc
        train_preds = predict(X_train, layers)
        train_acc = np.mean(np.round(train_preds) == y_train) * 100
        train_acc_history.append(train_acc)
        # testing acc
        test_preds = predict(X_test, layers)
        test_acc = np.mean(np.round(test_preds) == y_test) * 100
        test_acc_history.append(test_acc)    
    return layers, train_acc_history, test_acc_history

def predict(X, layers):
    set_training_mode(layers, training=False)
    out = X
    for layer in layers:
        out = layer.forward(out)
    return out
#running
my_layers = build_layers()
trained_layers, train_hist, test_hist = basic_fit(
    X_train, y_train, X_test, y_test, my_layers, epochs=15, lr=0.001, batch_size=32)
preds = predict(X_test, trained_layers)
accuracy = np.mean(np.round(preds) == y_test) * 100
print(f"Final test accuracy: {accuracy:.2f}%")

Starting training...
Epoch 1/15 | Train Acc: 57.69% | Test Acc: 50.48%
Epoch 2/15 | Train Acc: 80.37% | Test Acc: 63.58%
Epoch 3/15 | Train Acc: 90.71% | Test Acc: 66.77%
Epoch 4/15 | Train Acc: 91.67% | Test Acc: 66.13%
Epoch 5/15 | Train Acc: 83.33% | Test Acc: 66.13%
Epoch 6/15 | Train Acc: 83.33% | Test Acc: 66.45%
Epoch 7/15 | Train Acc: 79.49% | Test Acc: 68.05%
Epoch 8/15 | Train Acc: 93.43% | Test Acc: 68.37%
Epoch 9/15 | Train Acc: 94.95% | Test Acc: 69.65%
Epoch 10/15 | Train Acc: 95.91% | Test Acc: 68.69%
Epoch 11/15 | Train Acc: 98.08% | Test Acc: 71.57%
Epoch 12/15 | Train Acc: 98.72% | Test Acc: 73.16%
Epoch 13/15 | Train Acc: 99.20% | Test Acc: 71.25%
Epoch 14/15 | Train Acc: 99.44% | Test Acc: 73.16%
Epoch 15/15 | Train Acc: 99.76% | Test Acc: 74.12%

Final Test Accuracy: 74.12%
